# TP1 — Building a ReAct Agent from Scratch
## Module 2IAiasd 9.7 — Agentic AI — IMT Mines Alès

**Prerequisites:** You have completed TD1 (Exercises 0–4).  
Your Ollama server is still running and `qwen2.5:3b` is available.

**Objective:** Implement the full **ReAct loop** (Reason + Act) introduced in CM1,  
using the tools and concepts you practised in TD1.  
By the end of this session you will have a working agent that solves multi-step problems autonomously.



---

### Structure

| # | Exercise | Concepts |
|---|---------|----------|
| 1 | The ReAct loop — step by step | orchestrator, tool dispatch, ToolMessage |
| 2 | A complete math agent | multi-step reasoning, safety limit |
| 3 | Mixed-tool agent | combining tool types, robustness |
| 4 | Your own agent (open-ended) | design + evaluation |

> 📖 **Reference:** CM1 slides — perception → reasoning → action → observation loop  
> 📖 **Reference:** TD1 Exercise 4 — `@tool`, `bind_tools`, `ToolMessage`  


---
## Setup — Carry over from TD1

Run this cell to restore the TD1 environment.  
Make sure `ollama serve` is still running in your terminal.

In [18]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
import requests, os

MODEL = "qwen2.5:3b"

# Quick health check
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"✅ Ollama running | available models: {models}")
except Exception:
    print("❌ Ollama not reachable — run: ollama serve")

llm = ChatOllama(model=MODEL, temperature=0)
print(f"✅ LLM ready: {MODEL}")

# Optional: LangSmith tracing (from TD1)
# os.environ["LANGSMITH_API_KEY"]    = "lsv2_pt_..."
# os.environ["LANGSMITH_TRACING_V2"] = "true"
# os.environ["LANGSMITH_PROJECT"]    = "2IAiasd-9.7-TP1"

✅ Ollama running | available models: ['qwen2.5:3b', 'llama3.1:latest', 'nomic-embed-text:latest']
✅ LLM ready: qwen2.5:3b


---
## Exercise 1 — The ReAct Loop, Step by Step

In CM1 you saw the agent loop conceptually:
```
Perceive → Reason (LLM) → Act (tool call) → Observe (tool result) → loop
```

In TD1 Exercise 4 you made **one** tool call manually.  
Now you build the **full loop** that keeps running until the LLM decides it is done.

### 1.1 — Trace what the loop sees at each step

In [19]:
# Tools from TD1 Exercise 4 — reload them here
@tool
def get_word_count(text: str) -> int:
    """Count the number of words in a given text string."""
    return len(text.split())

@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature value from Celsius to Fahrenheit."""
    return round(celsius * 9 / 5 + 32, 2)

@tool
def get_current_date() -> str:
    """Return today's date in ISO format YYYY-MM-DD."""
    from datetime import date
    return date.today().isoformat()

@tool
def calculate_bmi(weight_kg: float, height_m: float) -> float:
    """Calculate Body Mass Index given weight in kilograms and height in metres."""
    return round(weight_kg / (height_m ** 2), 2)

tools_ex1 = [get_word_count, celsius_to_fahrenheit, get_current_date, calculate_bmi]
tools_map_ex1 = {t.name: t for t in tools_ex1}

llm_with_tools = llm.bind_tools(tools_ex1)
print(f"Available tools: {[t.name for t in tools_ex1]}")

Available tools: ['get_word_count', 'celsius_to_fahrenheit', 'get_current_date', 'calculate_bmi']


In [20]:
# Trace exactly what happens at each step for the query:
# "What is today's date and how many words are in 'Hello from IMT Mines Ales in France'?"

messages = [
    SystemMessage(content="You are a helpful assistant. Use tools when needed."),
    HumanMessage(content="What is today's date and how many words are in 'Hello from IMT Mines Ales in France'?"),
]

print("=" * 60)
print(f"INITIAL STATE: {len(messages)} messages")
for m in messages:
    print(f"  {m.type}: {m.content[:60]}")
print("=" * 60)

step = 0
while True:
    step += 1
    print(f"\n--- STEP {step}: REASON ---")

    response = llm_with_tools.invoke(messages)
    messages.append(response)

    print(f"  LLM content : {repr(response.content[:80]) if response.content else '(empty)'}")
    print(f"  Tool calls  : {response.tool_calls}")
    print(f"  Msg count   : {len(messages)}")

    if not response.tool_calls:
        print("\n--- LOOP EXITS: no more tool calls ---")
        break

    print(f"\n--- STEP {step}: ACT + OBSERVE ---")
    for tc in response.tool_calls:
        fn = tools_map_ex1[tc["name"]]
        result = fn.invoke(tc["args"])
        tool_msg = ToolMessage(content=str(result), tool_call_id=tc["id"])
        messages.append(tool_msg)
        print(f"  {tc['name']}({tc['args']}) → {result}")

print(f"\n{'=' * 60}")
print(f"FINAL ANSWER: {response.content}")
print(f"Total steps: {step} | Total messages: {len(messages)}")

INITIAL STATE: 2 messages
  system: You are a helpful assistant. Use tools when needed.
  human: What is today's date and how many words are in 'Hello from I

--- STEP 1: REASON ---
  LLM content : (empty)
  Tool calls  : [{'name': 'get_current_date', 'args': {}, 'id': '811c5a17-01ec-498c-a8f5-9d71d05231de', 'type': 'tool_call'}, {'name': 'get_word_count', 'args': {'text': 'Hello from IMT Mines Ales in France'}, 'id': '975257c0-82bc-4993-a512-668f48564a86', 'type': 'tool_call'}]
  Msg count   : 3

--- STEP 1: ACT + OBSERVE ---
  get_current_date({}) → 2026-10-18
  get_word_count({'text': 'Hello from IMT Mines Ales in France'}) → 7

--- STEP 2: REASON ---
  LLM content : "Today's date is 2026-10-18 and the sentence 'Hello from IMT Mines Ales in France"
  Tool calls  : []
  Msg count   : 6

--- LOOP EXITS: no more tool calls ---

FINAL ANSWER: Today's date is 2026-10-18 and the sentence 'Hello from IMT Mines Ales in France' contains 7 words.
Total steps: 2 | Total messages: 6


### 1.2 — ✏️ Your turn: trace analysis

Answer the following questions **in the Markdown cell below**,  
based on the trace you just observed. Be precise and reference specific message types.

**Answers (replace this text):**

1. How many tool calls did the model make? Which tools were called and in what order?
   > *two tools :  get_current_date and get_words_counts in order*

2. What does `response.content` contain when the model decides to call a tool vs. when it gives the final answer?
   > *it's empty when the model call the tools because the answers are in response.tool_calls.*

3. What is the stopping condition of the loop? Why is it reliable?
   > *when the llm doesn't need any more tools*

4. What would happen if you removed the `SystemMessage`? Try it and report.
   > *less presice answers*

---
## Exercise 2 — A Complete Math Agent

Now you build a **reusable `react_agent` function** that cleanly encapsulates the loop.  
This is exactly the pattern used by every production agent framework (LangGraph, CrewAI, AutoGen…).

### 2.1 — Define the math tools

In [21]:
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b. Raises ValueError if b is zero."""
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return round(a / b, 6)

@tool
def power(base: float, exponent: float) -> float:
    """Raise base to the power of exponent."""
    return base ** exponent

@tool
def square_root(x: float) -> float:
    """Return the square root of x. Raises ValueError if x is negative."""
    import math
    if x < 0:
        raise ValueError("Cannot compute the square root of a negative number.")
    return round(math.sqrt(x), 6)

math_tools = [add, subtract, multiply, divide, power, square_root]
print(f"Math tools: {[t.name for t in math_tools]}")

Math tools: ['add', 'subtract', 'multiply', 'divide', 'power', 'square_root']


### 2.2 — ✏️ Implement `react_agent`

Complete the function below. Requirements:
- Accepts `query` (str), `tools` (list), `system_prompt` (str, optional), `max_steps` (int, default 10), `verbose` (bool, default True)
- Returns the final answer as a string
- In verbose mode: prints each step's tool calls and observations
- Handles tool execution errors gracefully (catch exceptions, pass the error message as a `ToolMessage`)
- Stops if `max_steps` is reached and returns a timeout message

In [22]:
MY_SYSTEM_PROMPT = """
You are HealthTrack, a health and fitness calculator assistant for students and beginners.

Always use the provided tools for numerical calculations; never calculate tool outputs yourself.
Call exactly one tool per response. If a tool needs the result of another tool, wait for the real result before calling the next tool. Never invent intermediate values.
For BMI category questions, call bmi_category with the BMI returned by calculate_bmi.
If a tool returns an error, explain the valid input expected. If no calculation is needed, answer briefly without tools.
Answer in English in 2 to 4 short sentences, use metric units, and state that health estimates are not medical advice.
"""

def react_agent(
    query: str,
    tools: list,
    system_prompt: str = "You are a helpful assistant. Use tools when needed to answer accurately.",
    max_steps: int = 10,
    verbose: bool = True,
) -> str:
    # 1. Messages initiaux
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=query),
    ]

    # 2. Bind des outils
    llm_with_tools = llm.bind_tools(tools)

    # 3. Dictionnaire nom -> outil
    tools_map = {t.name: t for t in tools}

    # 4. Boucle ReAct, limitée à max_steps
    for step in range(1, max_steps + 1):
        # REASON
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if verbose:
            print(f"\n--- STEP {step} ---")
            print(f"  Tool calls: {response.tool_calls}")

        # Condition d'arrêt : plus d'appel d'outil -> réponse finale
        if not response.tool_calls:
            return response.content

        # ACT + OBSERVE
        for tc in response.tool_calls:
            # 5. Gestion des erreurs
            try:
                if tc["name"] not in tools_map:
                    raise KeyError(f"Unknown tool: {tc['name']}")
                result = tools_map[tc["name"]].invoke(tc["args"])
            except Exception as e:
                result = f"Error: {e}"

            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

            if verbose:
                print(f"  {tc['name']}({tc['args']}) -> {result}")

    # 6. max_steps atteint sans réponse finale
    return f"Timeout: no final answer after {max_steps} steps."

def run_my_agent(query: str) -> str:
    return react_agent(query=query, tools=my_agent_tools, system_prompt=MY_SYSTEM_PROMPT, max_steps=10, verbose=True)

In [23]:
# Test 1: simple two-step calculation
result = react_agent("What is (3 + 5) * 12?", math_tools)
print(f"\nResult: {result}")
if "96" not in result:
    # Retry with an explicit tool-use instruction for small models
    result = react_agent(
    "What is (3 + 5) * 12?",
    math_tools,
    system_prompt=(
        "You are a math assistant. Always use the provided tools for every arithmetic "
        "operation. Call only one tool at a time, wait for its result, then continue. "
        "When you have the final number, state it clearly."
    ),
)

print(result)


--- STEP 1 ---
  Tool calls: [{'name': 'add', 'args': {'a': 3, 'b': 5}, 'id': 'c3fd72e8-318e-4555-8472-497083bf240d', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 12, 'b': 'result_of_addition'}, 'id': '18309b8d-5ba7-4481-8285-65c077c41084', 'type': 'tool_call'}]
  add({'a': 3, 'b': 5}) -> 8.0
  multiply({'a': 12, 'b': 'result_of_addition'}) -> Error: 1 validation error for multiply
b
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='result_of_addition', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing

--- STEP 2 ---
  Tool calls: [{'name': 'add', 'args': {'a': 3, 'b': 5}, 'id': '1aeba957-279a-476f-8f3b-69e8e57a8fe9', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 12, 'b': 'result_of_addition'}, 'id': '6ff0aa3e-0e64-445a-9991-1761c01b6b89', 'type': 'tool_call'}]
  add({'a': 3, 'b': 5}) -> 8.0
  multiply({'a': 12, 'b': 'result_of_addition'}) -> Error: 1 validatio

In [8]:
# Test 2: multi-step with power and square root
result = react_agent(
    "Calculate: 2 to the power of 8, then subtract 56, then take the square root.",
    math_tools
)
print(f"\nResult: {result}")
# 2^8 = 256, 256 - 56 = 200, sqrt(200) ≈ 14.142
assert "14" in result, "Expected ~14.14 in the answer"
print("✅ Test 2 passed")


--- STEP 1 ---
  Tool calls: [{'name': 'power', 'args': {'base': 2, 'exponent': 8}, 'id': '0efe941f-cf11-4d12-8161-f0717528be76', 'type': 'tool_call'}]
  power({'base': 2, 'exponent': 8}) -> 256.0

--- STEP 2 ---
  Tool calls: [{'name': 'subtract', 'args': {'a': 256, 'b': 56}, 'id': 'c8464ec4-eaed-4e22-9b72-105b7b6820b3', 'type': 'tool_call'}]
  subtract({'a': 256, 'b': 56}) -> 200.0

--- STEP 3 ---
  Tool calls: [{'name': 'square_root', 'args': {'x': 200}, 'id': '5de689d6-0b16-4c57-9a7f-9aabae7247f2', 'type': 'tool_call'}]
  square_root({'x': 200}) -> 14.142136

--- STEP 4 ---
  Tool calls: []

Result: The final result after performing the operations is approximately 14.142136.
✅ Test 2 passed


In [9]:
# Test 3: error handling — division by zero
result = react_agent("What is 42 divided by 0?", math_tools, verbose=True)
print(f"\nResult: {result}")
# The agent should receive the error and report it gracefully without crashing
print("✅ Test 3 — verify the agent handled the error without raising an exception")


--- STEP 1 ---
  Tool calls: [{'name': 'divide', 'args': {'a': 42, 'b': 0}, 'id': '4c9b26c0-b2ee-4865-82a5-79e1563f43b1', 'type': 'tool_call'}]
  divide({'a': 42, 'b': 0}) -> Error: Cannot divide by zero.

--- STEP 2 ---
  Tool calls: []

Result: The operation of dividing 42 by 0 is not valid in mathematics because division by zero is undefined. Therefore, I cannot compute the result.
✅ Test 3 — verify the agent handled the error without raising an exception


In [14]:
# Test 4: no tool needed
result = react_agent("What is the Pythagorean theorem?", math_tools, verbose=False)
print(f"Result: {result}")
print("✅ Test 4 — agent answered without calling any tool")

Result: The Pythagorean theorem states that in a right-angled triangle, the square of the length of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the lengths of the other two sides. It can be written as: a² + b² = c², where c represents the length of the hypotenuse, and a and b represent the lengths of the other two sides.

I can help you perform calculations related to this theorem, if you need. For example, if you provide me with the lengths of two sides, I can calculate the length of the third side. Would you like to try that?
✅ Test 4 — agent answered without calling any tool


---
## Exercise 3 — Mixed-Tool Agent

A real-world agent needs tools of **different types** — not just one category.  
Here you combine the math tools from Exercise 2 with the utility tools from TD1,  
and add two new tools of your own design.

### 3.1 — Combine the existing tools

In [24]:
# Bring back the utility tools from TD1
combined_tools = math_tools + [get_word_count, celsius_to_fahrenheit, get_current_date, calculate_bmi]
print(f"Combined tools ({len(combined_tools)}): {[t.name for t in combined_tools]}")

Combined tools (10): ['add', 'subtract', 'multiply', 'divide', 'power', 'square_root', 'get_word_count', 'celsius_to_fahrenheit', 'get_current_date', 'calculate_bmi']


### 3.2 — ✏️ Add 2 new tools of your choice

Design and implement 2 new tools. Pick from the suggestions below or invent your own.  
Each tool must have:
- A clear, precise **docstring** (remember: the docstring IS the prompt sent to the LLM)
- **Type annotations** on all parameters
- At least one **edge case** handled (empty input, division by zero, negative value…)

**Suggestions:** `reverse_string`, `is_palindrome`, `count_vowels`, `days_until_date`, `percentage(value, total)`, `fibonacci(n)`, `is_prime(n)`

In [25]:
@tool
def count_vowels(text: str) -> int:
    """Count the number of vowels (a, e, i, o, u) in a given text."""
    return sum(1 for c in text.lower() if c in "aeiou")

@tool
def fibonacci(n: int) -> int:
    """Return the n-th Fibonacci number, where F(0)=0, F(1)=1, F(2)=1, F(3)=2. n must be >= 0."""
    if n < 0:
        raise ValueError("n must be a non-negative integer.")
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

# Add your tools to the combined list
all_tools = combined_tools + [count_vowels, fibonacci]
print(f"All tools ({len(all_tools)}): {[t.name for t in all_tools]}")

All tools (12): ['add', 'subtract', 'multiply', 'divide', 'power', 'square_root', 'get_word_count', 'celsius_to_fahrenheit', 'get_current_date', 'calculate_bmi', 'count_vowels', 'fibonacci']


### 3.3 — ✏️ Test with cross-category queries

Write **3 queries** that each require tools from **at least 2 different categories**  
(e.g. math + date, or string + math, or your custom tool + an existing one).

In [16]:
# Query 1: requires math + date (or math + another category)
# Example: "What is today's date, and what is 2 to the power of the day number?"
# TODO: write your own query
query1 = "TODO: what s 54 times 45 , and how much vowels the name idriss containts"
result1 = react_agent(query1, all_tools)
print(f"Q1: {query1}")
print(f"A1: {result1}\n")


--- STEP 1 ---
  Tool calls: [{'name': 'multiply', 'args': {'a': 54, 'b': 45}, 'id': 'a84e8e1d-df10-4cd8-a867-de0409ab79fa', 'type': 'tool_call'}, {'name': 'count_vowels', 'args': {'text': 'idriss'}, 'id': '3434ad34-acb0-4f73-af89-30328fda9b19', 'type': 'tool_call'}]
  multiply({'a': 54, 'b': 45}) -> 2430.0
  count_vowels({'text': 'idriss'}) -> 2

--- STEP 2 ---
  Tool calls: []
Q1: TODO: what s 54 times 45 , and how much vowels the name idriss containts
A1: The result of 54 times 45 is 2430.0. The name "idriss" contains 2 vowels.



In [26]:
# Query 2: requires one of your custom tools + an existing tool
# TODO: write your own query
query2 = "TODO: ehat is the fibonaci off 7 , and what is today date "
result2 = react_agent(query2, all_tools)
print(f"Q2: {query2}")
print(f"A2: {result2}\n")


--- STEP 1 ---
  Tool calls: [{'name': 'fibonacci', 'args': {'n': 7}, 'id': '695186b7-6a38-491c-96b4-9336c0537313', 'type': 'tool_call'}, {'name': 'get_current_date', 'args': {}, 'id': '97fdb22b-e8f2-4675-8195-60c7e51dcdc8', 'type': 'tool_call'}]
  fibonacci({'n': 7}) -> 13
  get_current_date({}) -> 2026-10-18

--- STEP 2 ---
  Tool calls: []
Q2: TODO: ehat is the fibonaci off 7 , and what is today date 
A2: The 7th Fibonacci number is 13. Today's date is October 18, 2026.



In [ ]:
# Query 3: a more complex chain (3+ tool calls expected)
query3 = ("Convert 25 degrees Celsius to Fahrenheit, count the vowels in the word 'agentic', "
          "then add those two results together.")
result3 = react_agent(query3, all_tools)
print(f"Q3: {query3}")
print(f"A3: {result3}\n")

### 3.4 — ✏️ Reflection

**Reflection (replace this text):**

1. Did the agent always call tools in the order you expected? If not, why?
   > *Your answer:Mostly. Independent calls (e.g. celsius_to_fahrenheit and count_vowels) were requested together in one step, which is fine. For dependent calls, qwen2.5:3b sometimes planned both steps at once and passed a placeholder ('result_of_addition') instead of waiting for the real result. This happens because a small model plans ahead without seeing intermediate results.*

2. Did the model use your custom tools correctly? What role did the docstring play?
   > *Your answer:The docstring, together with the name and type hints, is converted into the tool schema the LLM receives. It is the only description the model has, so a vague docstring (like my first TODO: version) leads to wrong or missed tool choices.*

3. Did any query cause a wrong tool call or a wrong answer? If so, how would you fix it?
   > *Your answer:Yes: in Exercise 2 the multiply call failed with a Pydantic validation error because the argument was a string. The try/except returned the error to the model as a ToolMessage, but the model repeated the mistake. Fixes: execute only one tool call per step so the model sees the real result first, write a stricter system prompt, or use a larger model such as llama3.1*

---
## Exercise 4 — Your Own Agent

Design and build a **domain-specific agent** of your choice.

**What to include:**
- At least **4 tools**, at least **2 original** (not reused from previous exercises)
- A carefully designed **system prompt**: role, constraints, output format
- At least **5 test queries**: normal cases, edge cases, and one no-tool case
- A brief **evaluation** of what worked and what didn't

### Domain ideas
- **Study assistant** — grade conversion (French 0–20 ↔ US A-F), GPA calculation, study-time estimator
- **Unit converter** — lengths, weights, temperatures, with context-aware explanations
- **Health tracker** — BMI, calorie estimation, heart-rate zones, VO2 max
- **Finance assistant** — interest calculation, savings projection, budget breakdown
- **Text analyser** — word count, reading time, keyword frequency, readability score
- **Your own idea** — describe it clearly in the design cell below

### 4.1 — ✏️ Agent design

Before writing any code, describe your agent here.

**Agent design (replace this text):**

**Name:*HealthTracker agent*

**Domain:*health and fitness*

**Who would use it and for what?**
> *Your answer:Students or beginners who want quick estimates of BMI, calorie needs and training zones.*

**List of tools (name + one-sentence description each):**
1. `tool_name_1` —calculate_bmi(wieght_kg,height_m):calculate body mass index 
2. `tool_name_2` —bmi_categorie(bmi):classifies as underwieght/normal/overwieght/obese
3. `tool_name_3` —estimate_bmr(weight_kg, height_cm, age, sex):Mifflin-St Jeor formula for resting calories
4. `tool_name_4` —daily_calorie_needs(bmr, activity_level):BMR times an activity factor (1.2 sedentary up to 1.9 very active)
5. `tool_name_5` —	heart_rate_zones(age):Max heart rate (220 − age) and the 5 training zones

**System prompt (draft):**
> *Your draft here:You are a health and fitness calculator assistant. Always use the provided tools for any calculation; never compute numbers yourself. Call one tool at a time and wait for its result before the next one. Use metric units (kg, cm, m). Give results with a short explanation and round to 1 decimal. Your results are estimates, not medical advice: for medical questions, recommend consulting a health professional. If the question needs no calculation, answer briefly without tools.*

**5 test queries you plan to run:**
Normal: “What is the BMI of someone who weighs 70 kg and is 1.75 m tall, and what category is that?”
Multi-step: “I’m a 25-year-old man, 70 kg, 175 cm, moderately active. How many calories should I eat per day?”
Edge case: “Calculate the heart rate zones for someone aged -5.” 
No tool: “What is a calorie?”
Your choice: “I’m 30, what are my heart rate zones for fat burning?”

### 4.2 — ✏️ Implement the tools

In [52]:
from typing import Literal

@tool
def calculate_bmi(weight_kg: float, height_m: float) -> float:
    """Calculate Body Mass Index (BMI) from weight in kilograms and height in METRES (e.g. 1.75, not 175).
    Returns ONLY the numeric BMI value, never the category. To get the category (underweight, normal,
    overweight, obese) you MUST call bmi_category afterwards with this value."""
    if weight_kg <= 0 or height_m <= 0:
        raise ValueError("weight_kg and height_m must be positive numbers.")
    if height_m > 2.5:
        raise ValueError("height_m looks like centimetres. Provide the height in metres (e.g. 1.75).")
    return round(weight_kg / height_m ** 2, 2)

@tool
def bmi_category(bmi: float) -> str:
    """Return the WHO weight category (underweight, normal, overweight or obese) for a given BMI value.
    Use it after calculate_bmi whenever the user asks for a category."""
    if bmi <= 0:
        raise ValueError("bmi must be a positive number.")
    if bmi < 18.5:
        return "underweight"
    if bmi < 25:
        return "normal"
    if bmi < 30:
        return "overweight"
    return "obese"

@tool
def estimate_bmr(weight_kg: float, height_cm: float, age: int, sex: Literal["male", "female"]) -> float:
    """Estimate the Basal Metabolic Rate (calories burned at rest per day, in kcal) with the Mifflin-St Jeor formula.
    Height must be in CENTIMETRES (e.g. 175). Use it before daily_calorie_needs."""
    if weight_kg <= 0:
        raise ValueError("weight_kg must be positive.")
    if not 100 <= height_cm <= 250:
        raise ValueError("height_cm must be between 100 and 250 (centimetres).")
    if not 1 <= age <= 120:
        raise ValueError("age must be between 1 and 120 years.")
    base = 10 * weight_kg + 6.25 * height_cm - 5 * age
    return round(base + (5 if sex == "male" else -161), 2)

@tool
def daily_calorie_needs(
    bmr: float,
    activity_level: Literal["sedentary", "light", "moderate", "active", "very_active"],
) -> float:
    """Estimate total daily calorie needs (kcal/day) by multiplying the BMR by an activity factor.
    activity_level: sedentary (little or no exercise), light (1-3 days/week), moderate (3-5 days/week),
    active (6-7 days/week), very_active (physical job or twice-daily training).
    Requires the BMR value returned by estimate_bmr."""
    if bmr <= 0:
        raise ValueError("bmr must be a positive number.")
    factors = {"sedentary": 1.2, "light": 1.375, "moderate": 1.55, "active": 1.725, "very_active": 1.9}
    return round(bmr * factors[activity_level], 2)

@tool
def heart_rate_zones(age: int) -> str:
    """Compute the maximum heart rate (220 minus age) and the 5 training zones in beats per minute for a given age in years."""
    if not 1 <= age <= 120:
        raise ValueError("age must be between 1 and 120 years.")
    max_hr = 220 - age
    zones = [
        ("Zone 1 (very light, warm-up)", 0.50, 0.60),
        ("Zone 2 (light, fat burning)", 0.60, 0.70),
        ("Zone 3 (moderate, aerobic)", 0.70, 0.80),
        ("Zone 4 (hard, threshold)", 0.80, 0.90),
        ("Zone 5 (maximum effort)", 0.90, 1.00),
    ]
    lines = [f"Max heart rate: {max_hr} bpm"]
    lines += [f"{name}: {round(max_hr * lo)}-{round(max_hr * hi)} bpm" for name, lo, hi in zones]
    return "\n".join(lines)

my_agent_tools = [calculate_bmi, bmi_category, estimate_bmr, daily_calorie_needs, heart_rate_zones]
print(f"{len(my_agent_tools)} tools: {[t.name for t in my_agent_tools]}")

5 tools: ['calculate_bmi', 'bmi_category', 'estimate_bmr', 'daily_calorie_needs', 'heart_rate_zones']


### 4.3 — ✏️ Run the 5 test queries

In [53]:
def react_agent(
    query: str,
    tools: list,
    system_prompt: str = "You are a helpful assistant. Use tools when needed to answer accurately.",
    max_steps: int = 10,
    verbose: bool = True,
) -> str:
    """Minimal ReAct agent: reason -> act -> observe, one tool call executed per step."""
    messages = [SystemMessage(content=system_prompt), HumanMessage(content=query)]
    llm_with_tools = llm.bind_tools(tools)
    tools_map = {t.name: t for t in tools}
    deferred_tools = []  # names of tool calls skipped and still to be re-requested

    for step in range(1, max_steps + 1):
        # REASON
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if verbose:
            print(f"\n--- STEP {step} ---")
            print(f"  Tool calls: {response.tool_calls}")

        # No tool call: final answer, unless a skipped call is still pending
        if not response.tool_calls:
            if deferred_tools:
                required = ", ".join(deferred_tools)
                messages.append(HumanMessage(
                    content=f"Do not answer yet. You must call {required} using the exact result from the previous tool."
                ))
                if verbose:
                    print(f"  Final answer rejected; waiting for tool call(s): {required}")
                continue
            return response.content

        # ACT + OBSERVE
        for i, tc in enumerate(response.tool_calls):
            # Execute only the first call of each step, so dependent calls see the real result
            if i > 0:
                deferred_tools.append(tc["name"])
                messages.append(ToolMessage(
                    content="This call was not executed because only one tool call is allowed per turn. "
                            "Call this tool again in the next turn using the exact value returned by "
                            "the previous tool; do not answer yet.",
                    tool_call_id=tc["id"],
                ))
                if verbose:
                    print(f"  {tc['name']}({tc['args']}) -> skipped")
                continue

            try:
                if tc["name"] not in tools_map:
                    raise KeyError(f"Unknown tool: {tc['name']}")
                result = tools_map[tc["name"]].invoke(tc["args"])
            except Exception as e:
                result = f"Error: {e}"

            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
            if tc["name"] in deferred_tools:
                deferred_tools.remove(tc["name"])
            if verbose:
                print(f"  {tc['name']}({tc['args']}) -> {result}")

    return f"Timeout: no final answer after {max_steps} steps."

In [54]:
print(calculate_bmi.invoke({"weight_kg": 70, "height_m": 1.75}))                              # 22.86
print(bmi_category.invoke({"bmi": 22.86}))                                                    # normal
print(estimate_bmr.invoke({"weight_kg": 70, "height_cm": 175, "age": 25, "sex": "male"}))     # 1673.75
print(daily_calorie_needs.invoke({"bmr": 1673.75, "activity_level": "moderate"}))             # 2594.31
print(heart_rate_zones.invoke({"age": 30}))                                                   # max 190, Zone 2 = 114-133

22.86
normal
1673.75
2594.31
Max heart rate: 190 bpm
Zone 1 (very light, warm-up): 95-114 bpm
Zone 2 (light, fat burning): 114-133 bpm
Zone 3 (moderate, aerobic): 133-152 bpm
Zone 4 (hard, threshold): 152-171 bpm
Zone 5 (maximum effort): 171-190 bpm


In [55]:
MY_SYSTEM_PROMPT = """
You are HealthTrack, a health and fitness calculator assistant for students and beginners.

Always use the provided tools for numerical calculations; never calculate tool outputs yourself.
Call exactly one tool per response. If a tool needs the result of another tool, wait for the real result before calling the next tool. Never invent intermediate values.
If the user asks for a BMI category (or whether a BMI is normal), you MUST call bmi_category with the BMI returned by calculate_bmi. Never state a category without calling it.
When heart_rate_zones returns the zones, report the maximum heart rate and all 5 zones.
If a tool returns an error, explain the valid input expected. If no calculation is needed, answer briefly without tools.
Answer in English, in short sentences, use metric units, and state that health estimates are not medical advice.
"""

def run_my_agent(query: str) -> str:
    return react_agent(
        query=query,
        tools=my_agent_tools,
        system_prompt=MY_SYSTEM_PROMPT,
        max_steps=10,
        verbose=True,
    )

In [56]:
# Query 1 — normal case
q1 = "What is the BMI of someone who weighs 70 kg and is 1.75 m tall, and what category is that?"
print(f"Query 1: {q1}")
r1 = run_my_agent(q1)
print(f"Answer: {r1}\n")

Query 1: What is the BMI of someone who weighs 70 kg and is 1.75 m tall, and what category is that?

--- STEP 1 ---
  Tool calls: [{'name': 'calculate_bmi', 'args': {'weight_kg': 70, 'height_m': 1.75}, 'id': '77a6e40c-d0ad-4aca-9f6e-21da57cf0c8f', 'type': 'tool_call'}]
  calculate_bmi({'weight_kg': 70, 'height_m': 1.75}) -> 22.86

--- STEP 2 ---
  Tool calls: [{'name': 'bmi_category', 'args': {'bmi': 22.86}, 'id': '06dbffec-9b44-4b45-ad84-eb6a3ab9b9cb', 'type': 'tool_call'}]
  bmi_category({'bmi': 22.86}) -> normal

--- STEP 3 ---
  Tool calls: []
Answer: The BMI category for 22.86 is normal.



In [57]:
# Query 2 — multi-step
q2 = "I'm a 25-year-old man, 70 kg, 175 cm, moderately active. How many calories should I eat per day?"
print(f"Query 2: {q2}")
r2 = run_my_agent(q2)
print(f"Answer: {r2}\n")

Query 2: I'm a 25-year-old man, 70 kg, 175 cm, moderately active. How many calories should I eat per day?

--- STEP 1 ---
  Tool calls: [{'name': 'estimate_bmr', 'args': {'sex': 'male', 'weight_kg': 70, 'height_cm': 175, 'age': 25}, 'id': 'fc674638-7f35-4f39-84b2-5f91356111f3', 'type': 'tool_call'}, {'name': 'daily_calorie_needs', 'args': {'bmr': 1852.5, 'activity_level': 'moderate'}, 'id': '75a1c69b-635c-4ae2-9dac-23d323a191b9', 'type': 'tool_call'}]
  estimate_bmr({'sex': 'male', 'weight_kg': 70, 'height_cm': 175, 'age': 25}) -> 1673.75
  daily_calorie_needs({'bmr': 1852.5, 'activity_level': 'moderate'}) -> skipped

--- STEP 2 ---
  Tool calls: [{'name': 'daily_calorie_needs', 'args': {'activity_level': 'moderate', 'bmr': 1673.75}, 'id': '5c81e0d3-6e23-4069-ac2f-7f392b065404', 'type': 'tool_call'}]
  daily_calorie_needs({'activity_level': 'moderate', 'bmr': 1673.75}) -> 2594.31

--- STEP 3 ---
  Tool calls: []
Answer: Based on your information, you should aim to eat approximately 259

In [58]:
# Query 3 — edge case
q3 = "Calculate the heart rate zones for someone aged -5."
print(f"Query 3: {q3}")
r3 = run_my_agent(q3)
print(f"Answer: {r3}\n")

Query 3: Calculate the heart rate zones for someone aged -5.

--- STEP 1 ---
  Tool calls: [{'name': 'heart_rate_zones', 'args': {'age': -5}, 'id': 'df8a3da6-f077-4a1a-86a4-1cbf31548345', 'type': 'tool_call'}]
  heart_rate_zones({'age': -5}) -> Error: age must be between 1 and 120 years.

--- STEP 2 ---
  Tool calls: []
Answer: It seems there was an error with the age provided. Age must be between 1 and 120 years. Please provide a valid age.



In [59]:
# Query 4 — no tool needed
q4 = "What is a calorie?"
print(f"Query 4: {q4}")
r4 = run_my_agent(q4)
print(f"Answer: {r4}\n")

Query 4: What is a calorie?

--- STEP 1 ---
  Tool calls: []
Answer: A calorie is a unit of energy that is used to measure the energy content in food.



In [60]:
# Query 5 — your choice
q5 = "I'm 30 years old. What are my heart rate zones, and which one is best for fat burning?"
print(f"Query 5: {q5}")
r5 = run_my_agent(q5)
print(f"Answer: {r5}\n")

Query 5: I'm 30 years old. What are my heart rate zones, and which one is best for fat burning?

--- STEP 1 ---
  Tool calls: [{'name': 'heart_rate_zones', 'args': {'age': 30}, 'id': 'ab72c389-ea4f-4609-a33f-1fc0615ce118', 'type': 'tool_call'}]
  heart_rate_zones({'age': 30}) -> Max heart rate: 190 bpm
Zone 1 (very light, warm-up): 95-114 bpm
Zone 2 (light, fat burning): 114-133 bpm
Zone 3 (moderate, aerobic): 133-152 bpm
Zone 4 (hard, threshold): 152-171 bpm
Zone 5 (maximum effort): 171-190 bpm

--- STEP 2 ---
  Tool calls: []
Answer: Your heart rate zones are as follows:
- Zone 1 (very light, warm-up): 95-114 bpm
- Zone 2 (light, fat burning): 114-133 bpm
- Zone 3 (moderate, aerobic): 133-152 bpm
- Zone 4 (hard, threshold): 152-171 bpm
- Zone 5 (maximum effort): 171-190 bpm

The fat burning zone for you is Zone 2, which is 114-133 bpm. This is the best for fat burning.



### 4.4 — ✏️ Evaluation

Fill in the table and answer the questions.

**Evaluation table:**

| Query | Tools called | Correct? | Notes |
|-------|-------------|----------|-------|
| Q1 | calculate_bmi → bmi_category | ✅ | BMI 22.86, normal category. |
| Q2 | estimate_bmr → daily_calorie_needs | ✅ after the loop fix | BMR 1673.75; daily needs ≈ 2594.31 kcal, not 2700. |
| Q3 | heart_rate_zones | ✅ | Invalid age is rejected by the tool. |
| Q4 | none | ✅ | A definition does not require a calculation tool. |
| Q5 | heart_rate_zones | ✅ | Zone 2 is approximately 114–133 bpm for fat-burning training. |

**Questions:**

1. What is the most important thing you learned about writing effective tool docstrings?
   > *Your answer:*

2. Where did your agent fail or surprise you? What would you change?
   > *Your answer:*

3. How does this ReAct agent relate to the four agentic patterns from CM1?
   > *Your answer:*

4. In TP2 your agent will retrieve information from documents (RAG) instead of calling Python functions.  
   What do you think will change in the loop? What stays the same?
   > *Your answer:*

---
## What comes next

| Session | Date | You will build |
|---------|------|----------------|
| TP2 | Nov 10–13 | RAG pipeline — retrieval as a tool, ChromaDB, RAGAS evaluation |
| TP3+TP4 | Nov 18 | Multi-agent system with CrewAI + intro to LangGraph |
| EDT1–4 | Nov 23 → Dec 8 | Your group project — agent on a real business problem |
| Soutenance | Dec 9 | Demo + Q&A |
| QCM | Dec 10 | 1h — key concepts from the module |